In [1]:
pip install datasets

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.25a0+6627725-py3.12-linux-x86_64.egg is deprecated. pip 25.1 wi

In [11]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [12]:
example = dataset["train"][0]

print("QUESTION:")
print(example["question"])

print("\nANSWER + ORIGINAL REASONING:")
print(example["answer"])

QUESTION:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

ANSWER + ORIGINAL REASONING:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [14]:
import re

def extract_final_answer(answer):
    match = re.search(r"####\s*(.*)", answer)

    if match:
        return match.group(1).strip()

    return None

In [15]:
answer = dataset["train"][0]["answer"]

final_answer = extract_final_answer(answer)

print(final_answer)

72


In [17]:
#question
##correct_answer

In [18]:
def prepare_example(example):
    return {
        "question": example["question"],
        "correct_answer": extract_final_answer(example["answer"])
    }

teacher_dataset = dataset.map(prepare_example)

Map: 100%|██████████| 1319/1319 [00:00<00:00, 53556.06 examples/s]


In [19]:
print(teacher_dataset["train"][0])

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'correct_answer': '72'}


In [20]:
teacher_dataset.save_to_disk("gsm8k_teacher_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 1319/1319 [00:00<00:00, 329165.64 examples/s]


In [4]:
DATASET_DIR = Path("gsm8k dataset")

In [ ]:
DATASET_DIR.mkdir(exist_ok=True)
def extract_answer(answer):
    match = re.search(r"####\s*(.*)", answer)

    if match:
        return match.group(1).strip()

    return None
    

In [4]:

import json
import re
import urllib.request

# Direct GSM8K dataset URLs
BASE_URL = (
    "https://raw.githubusercontent.com/openai/grade-school-math/"
    "master/grade_school_math/data/"
)

FILES = {
    "train": BASE_URL + "train.jsonl",
    "test": BASE_URL + "test.jsonl"
}


def download_and_convert(url, split_name):
    print(f"Downloading {split_name} dataset...")

    # Download without using the datasets or numpy libraries
    with urllib.request.urlopen(url, timeout=60) as response:
        content = response.read().decode("utf-8")

    converted_data = []

    for i, line in enumerate(content.splitlines()):
        if not line.strip():
            continue

        item = json.loads(line)

        full_answer = item["answer"]

        # Separate reasoning and final answer at ####
        parts = re.split(r"\n####\s*", full_answer, maxsplit=1)

        reasoning = parts[0].strip()
        ans = parts[1].strip() if len(parts) > 1 else ""

        # Remove GSM8K calculator annotations while retaining expressions
        reasoning = re.sub(r"<<(.*?)>>", r"\1", reasoning)

        converted_data.append({
            "qid": i+1,
            "question": item["question"],
            "reasoning": reasoning,
            "ans": ans
        })

    output_file = f"gsm8k_{split_name}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(converted_data, f, indent=2, ensure_ascii=False)

    print(f"Saved: {output_file}")
    print(f"Examples: {len(converted_data)}")

    return converted_data


# Download and convert both splits
train_data = download_and_convert(FILES["train"], "train")
test_data = download_and_convert(FILES["test"], "test")

print("\nFirst training example:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))


Saved: gsm8k_train.json
Examples: 7473
Saved: gsm8k_test.json
Examples: 1319

First training example:
{
  "qid": 1,
  "question": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
  "reasoning": "Natalia sold 48/2 = 48/2=2424 clips in May.\nNatalia sold 48+24 = 48+24=7272 clips altogether in April and May.",
  "ans": "72"
}
